In [1]:
!pip install nltk gensim


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import nltk
import re

from nltk.corpus import gutenberg
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download("gutenberg")
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
shakespeare_files = [
    file for file in gutenberg.fileids()
    if file.startswith("shakespeare")
]

print(shakespeare_files)

['shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt']


In [4]:
hamlet = gutenberg.raw("shakespeare-hamlet.txt")
hamlet = hamlet[:10000]
print(hamlet[:100])

[The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo a


In [5]:
# 1. Lowercase
text = hamlet.lower()
# 2. Remove punctuation/special characters
text = re.sub(r'[^a-zA-Z\s]', '', text)
# 3. Remove extra spaces
text = re.sub(r'\s+', ' ', text).strip()

print(text[:1000])

the tragedie of hamlet by william shakespeare actus primus scoena prima enter barnardo and francisco two centinels barnardo whos there fran nay answer me stand vnfold your selfe bar long liue the king fran barnardo bar he fran you come most carefully vpon your houre bar tis now strook twelue get thee to bed francisco fran for this releefe much thankes tis bitter cold and i am sicke at heart barn haue you had quiet guard fran not a mouse stirring barn well goodnight if you do meet horatio and marcellus the riuals of my watch bid them make hast enter horatio and marcellus fran i thinke i heare them stand whos there hor friends to this ground mar and leigemen to the dane fran giue you good night mar o farwel honest soldier who hath relieud you fra barnardo has my place giue you goodnight exit fran mar holla barnardo bar say what is horatio there hor a peece of him bar welcome horatio welcome good marcellus mar what has this thing appeard againe to night bar i haue seene nothing mar horati

In [6]:
# Word Tokenization
tokens = word_tokenize(text)

print("Tokens:")
print(tokens[:100])

# Stopword Removal
stop_words = set(stopwords.words('english'))

filtered_tokens = [word for word in tokens if word not in stop_words]

print("\nAfter Stopword Removal:")
print(filtered_tokens[:100])

Tokens:
['the', 'tragedie', 'of', 'hamlet', 'by', 'william', 'shakespeare', 'actus', 'primus', 'scoena', 'prima', 'enter', 'barnardo', 'and', 'francisco', 'two', 'centinels', 'barnardo', 'whos', 'there', 'fran', 'nay', 'answer', 'me', 'stand', 'vnfold', 'your', 'selfe', 'bar', 'long', 'liue', 'the', 'king', 'fran', 'barnardo', 'bar', 'he', 'fran', 'you', 'come', 'most', 'carefully', 'vpon', 'your', 'houre', 'bar', 'tis', 'now', 'strook', 'twelue', 'get', 'thee', 'to', 'bed', 'francisco', 'fran', 'for', 'this', 'releefe', 'much', 'thankes', 'tis', 'bitter', 'cold', 'and', 'i', 'am', 'sicke', 'at', 'heart', 'barn', 'haue', 'you', 'had', 'quiet', 'guard', 'fran', 'not', 'a', 'mouse', 'stirring', 'barn', 'well', 'goodnight', 'if', 'you', 'do', 'meet', 'horatio', 'and', 'marcellus', 'the', 'riuals', 'of', 'my', 'watch', 'bid', 'them', 'make', 'hast']

After Stopword Removal:
['tragedie', 'hamlet', 'william', 'shakespeare', 'actus', 'primus', 'scoena', 'prima', 'enter', 'barnardo', 'francisc

## Word2Vec

In [7]:
from gensim.models import Word2Vec

# Word2Vec expects a list of tokenized sentences
tokenized_text = [filtered_tokens]

w2v_model = Word2Vec(
    sentences=tokenized_text,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=100
)

word = "hamlet"

if word in w2v_model.wv:
    vector = w2v_model.wv[word]

    print("Word:", word)
    print("Vector shape:", vector.shape)
    print("First 10 values:", vector[:10])

    print("\nSimilar words:")
    print(w2v_model.wv.most_similar(word, topn=5))
else:
    print(f"'{word}' is not present in the vocabulary.")
    print("Try one of these words:")
    print(list(w2v_model.wv.index_to_key[:10]))

Word: hamlet
Vector shape: (100,)
First 10 values: [-0.08896035  0.11049638  0.13065849 -0.01077442 -0.48750013 -0.31913227
  0.13012172  0.32148612 -0.02349658  0.2224675 ]

Similar words:
[('side', 0.8541527986526489), ('knowne', 0.8504745364189148), ('world', 0.843991219997406), ('tragedie', 0.8410220742225647), ('combate', 0.8228403329849243)]


## GLOVE

In [ ]:
import gensim.downloader as api

# Load pretrained GloVe model
glove = api.load("glove-wiki-gigaword-100")

# Get GloVe vector for each word
glove_vectors = []

for word in filtered_tokens:
    if word in glove:
        glove_vectors.append(glove[word])

print("\nNumber of vectors:", len(glove_vectors))

# First word and its vector
print("\nFirst word:", filtered_tokens[0])
print("GloVe vector:")
print(glove_vectors[0])

print("\nVector dimensions:", glove_vectors[0].shape)

## BERT

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased')
bert_model.eval()

# Prepare the input text for BERT
inputs = bert_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512,
    padding=True
)

# Generate BERT embeddings
with torch.no_grad():
    outputs = bert_model(**inputs)

# Token-level embeddings
token_embeddings = outputs.last_hidden_state

print("Token embedding shape:", token_embeddings.shape)
# Shape: [1, number_of_tokens, 768]

# Create one sentence/document embedding using mean pooling
attention_mask = inputs["attention_mask"].unsqueeze(-1)

masked_embeddings = token_embeddings * attention_mask
sentence_embedding = masked_embeddings.sum(dim=1) / attention_mask.sum(dim=1)

print("Sentence embedding shape:", sentence_embedding.shape)
print("First 10 values:")
print(sentence_embedding[0][:10])

c:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2236.31it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading 

Token embedding shape: torch.Size([1, 512, 768])
Sentence embedding shape: torch.Size([1, 768])
First 10 values:
tensor([-0.5064,  0.3178,  0.3201, -0.2921,  0.2475, -0.2163,  0.4437,  0.1912,
         0.0112, -0.4871])
